In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import napari
import colorcet as cc
import pandas as pd

import dnt

spots_directory = Path(r"C:\Tracking\BlastodermAnalysis\data\spots")
save_path = Path(r"C:\Tracking\BlastodermAnalysis\figures\output\displacements")
save_path.mkdir(parents=True, exist_ok=True)

include = [1, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14]
condition = [0, 0, 0, 0, 0, 1, 1, 2, 2, 2, 2]
earliest_frames = [25, 43, 63, 80, 75, 150, 23, 36, 17, 70, 3]

dnt.set_plot_style()

spots_dfs, metadatas, stems = dnt.load_spots_data(spots_directory, include)

df = spots_dfs[0]
cycles = [10, 11, 12, 13, 14]

print(df.columns)

In [ ]:
from collections import defaultdict

displacements = defaultdict(list)

for k, df in enumerate(spots_dfs):

    if condition[k] != 0:
        continue

    df = df[df["frame"] > earliest_frames[k]]

    min_mvmt_frames, times = dnt.find_stationary_timepoints(df)

    tdf = dnt.generate_timepoint_df(df)
    tdf_n_tracklets = tdf.groupby("track_id")["x"].count() == 31
    print(tdf_n_tracklets.mean())
    included_tracklets = tdf["track_id"].map(tdf_n_tracklets)

    if tdf_n_tracklets.sum() < 100:
        continue

    tdf = tdf[included_tracklets]

    track_tdf = tdf.groupby(["track_id", "cycle"])[["z", "y", "x", "AP"]].mean()

    for col in ["x", "y", "z", "AP"]:
        track_tdf[f"d{col}"] = track_tdf.groupby(level="track_id")[col].diff()

    # sns.violinplot(track_tdf, x="cycle", y="dAP")
    # plt.show()

    track_tdf["displacement"] = np.sqrt(track_tdf["dx"]**2 + track_tdf["dy"]**2 + track_tdf["dz"]**2)
    track_tdf = track_tdf.reset_index()

    for cycle in [11, 12, 13, 14]:
        cycle_subset = track_tdf[track_tdf["cycle"] == cycle]
        displacements["cycle"].append(cycle)
        displacements["embryo"].append(k)
        displacements["mean_displacement"].append(np.mean(cycle_subset["displacement"]))




    # print(track_tdf)
    # sns.barplot(track_tdf, x="cycle", y="displacement")
    # plt.show()

In [ ]:
displacements = pd.DataFrame(displacements)
fig, ax = plt.subplots(1, 1, figsize=(3.3, 4))
sns.barplot(displacements, x="cycle", y="mean_displacement", hue="cycle", palette=dnt.palettes.nc, edgecolor="k", legend=False)
sns.stripplot(displacements, x="cycle", y="mean_displacement", color="k")
plt.xlabel("Nuclear Cycle")
plt.ylabel("Avg displacement (um)")
plt.title("Average displacement of lineages")
plt.tight_layout()
plt.savefig(save_path / "lineage_displacements.png")
plt.show()